# Task 3: Merge - recursive learning, then distillation
Run the Task 1 loop to obtain a final teacher, distill it into the student (hard labels only on the seed set), and ask whether the loop's gain survives distillation.

In [1]:
import sys, pathlib
ROOT = pathlib.Path.cwd()
while not (ROOT / "configs").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from src.utils import load_config
from src.data import load_base, make_split
from src.features import attach_views

DATA_SOURCE = "local"   # "local" (data/raw TSVs), "hf", or "synthetic" (smoke test only)
cfg = load_config(overrides={"data.source": DATA_SOURCE})
base = attach_views(load_base(cfg), cfg)
cfg["_n_classes"] = base.n_classes
print(base.label_names, {k: len(v) for k, v in base.texts.items()})

['business', 'health', 'politics', 'sports'] {'train': 1311, 'dev': 188, 'test': 376}


In [2]:
import json
from task3_merged import run as t3
print('see task3_merged/run.py for the full pipeline')
df = t3.main(['--seeds', '0', '--out', 'results/task3_notebook', '--set', f'data.source={DATA_SOURCE}'])

see task3_merged/run.py for the full pipeline
data source=local backend=tfidf classes=['business', 'health', 'politics', 'sports'] sizes={'train': 1311, 'dev': 188, 'test': 376}
using (T, alpha) = (4.0, 0.5) selected on dev in Task 2
  [seed 0] round 0 baseline dev macro-F1=0.7892
  [seed 0] round 1: cand=20 precision=1.000 dev F1 0.7962 -> accepted (current 0.7962)
  [seed 0] round 2: cand=36 precision=1.000 dev F1 0.8092 -> accepted (current 0.8092)
  [seed 0] round 3: cand=61 precision=0.984 dev F1 0.7392 -> rejected (current 0.8092)
  [seed 0] round 4: cand=61 precision=0.984 dev F1 0.7293 -> rejected (current 0.8092)
  [seed 0] stopping: 2 consecutive rejections
  [seed 0] done: teacher F1 r0->final 0.8435->0.8456

=== Task 3: TEST results (mean/std over seeds) ===
                               test_macro_f1     test_ece         params     latency_ms    
                                        mean std     mean std       mean std       mean std
model                              

In [3]:
gain = pd.read_csv(ROOT / 'results/task3_notebook/gain_survival.csv'); gain

,seed,teacher_gain_test,student_gain_test,teacher_gain_dev,student_gain_dev
0,0,0.002071,0.001264,0.019978,0.016803


In [4]:
df.groupby('model')[['test_macro_f1', 'test_ece', 'params', 'latency_ms']].mean().round(4)

,test_macro_f1,test_ece,params,latency_ms
model,,,,
student_from_final,0.7961,0.1977,8004.0,0.0309
student_from_round0,0.7949,0.2483,8004.0,0.0271
student_hard_pseudo,0.7918,0.2185,8004.0,0.0276
student_seed_hard,0.8095,0.2796,8004.0,0.0354
student_task2_full_labels(ref),0.9062,0.0293,8004.0,0.0416
teacher_final_loop,0.8456,0.2474,2561284.0,0.1408
teacher_full_labels(ref),0.9027,0.0384,2561284.0,0.1592
teacher_round0,0.8435,0.3109,2561284.0,0.1324


Multi-seed results: `python -m task3_merged.run` (writes `results/task3/`). A null result (the gain does not survive distillation) is a valid finding.